# Práctica 10 — Integración de muestras y análisis pseudobulk: PBMC estimulados con IFN-β

**Máster en Bioinformática — Módulo de Transcriptómica**
**Autor:** Víctor Sánchez-Arévalo · [github.com/victorsanchezarevalo/transcriptomica-master](https://github.com/victorsanchezarevalo/transcriptomica-master)

---

En la práctica 9 analizamos **una** muestra de célula única. La pregunta experimental habitual, sin embargo, es comparativa: **¿cómo responden las células a un estímulo?** Y ahí aparecen dos problemas nuevos:

1. **Integración:** al combinar muestras procesadas por separado, las diferencias técnicas (efecto batch) pueden confundirse con biología.
2. **Réplicas:** con 6.000 células por grupo, un test célula a célula las trata como 6.000 réplicas independientes — pero no lo son: la réplica biológica real es el **donante**.

**Dataset:** [Kang *et al.*, Nature Biotechnology 2018](https://doi.org/10.1038/nbt.4042) (GEO: [GSE96583](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE96583)). PBMC de **8 donantes** multiplexados genéticamente en dos corridas de 10x: cultivados 6 h **sin** (`ctrl`) o **con** interferón-β (`stim`). La asignación de cada célula a su donante se hizo por variantes genéticas (*demuxlet*); nosotros usaremos la anotación de los autores.

**Objetivos de aprendizaje:**

1. Construir un objeto `AnnData` multimuestra desde ficheros crudos de GEO.
2. Visualizar y corregir el efecto batch con **Harmony**.
3. Entender la falacia de las pseudoréplicas y aplicar el análisis **pseudobulk** con **PyDESeq2** (recuperando las prácticas 3–4).
4. Conectar la respuesta al interferón con el análisis funcional de la práctica 7.

> **Tiempo estimado:** una sesión de 2 horas.

## 0. Instalación y carga de librerías

Además de scanpy necesitamos:

- **harmonypy** (fijamos la versión 0.0.10, la compatible con el *wrapper* de scanpy): integración de muestras con Harmony (Korsunsky *et al.*, Nature Methods 2019).
- **pydeseq2**: el DESeq2 que ya conocéis de las prácticas 3–4, para el análisis pseudobulk.
- **decoupler**: para la sección opcional de análisis funcional (práctica 7).

In [ ]:
!pip install -q scanpy leidenalg "harmonypy==0.0.10" pydeseq2 decoupler

In [ ]:
import importlib.metadata

import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp
import scanpy as sc
import anndata as ad

sc.settings.verbosity = 1

for pkg in ["scanpy", "anndata", "harmonypy", "pydeseq2", "decoupler"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")

## 1. Descarga de los datos desde GEO

Descargamos directamente del GEO (NCBI) los ficheros de las dos corridas 10x (`2.1` = control, `2.2` = estimulada) más la tabla de metadatos de los autores:

| Fichero | Contenido |
|---|---|
| `GSM2560248_2.1.mtx.gz` | matriz de conteos, corrida control (14.619 células) |
| `GSM2560249_2.2.mtx.gz` | matriz de conteos, corrida estimulada (14.446 células) |
| `*_barcodes.tsv.gz` | códigos de barras de cada corrida |
| `GSE96583_batch2.genes.tsv.gz` | anotación de genes (Ensembl + símbolo) |
| `GSE96583_batch2.total.tsne.df.tsv.gz` | metadatos de los autores: donante, condición, tipo celular, dobletes |

In [ ]:
import urllib.request
from pathlib import Path

DATA_DIR = Path("datos_kang2018")
DATA_DIR.mkdir(exist_ok=True)

BASE = "https://ftp.ncbi.nlm.nih.gov/geo"
FICHEROS = {
    "2.1.mtx.gz":        f"{BASE}/samples/GSM2560nnn/GSM2560248/suppl/GSM2560248_2.1.mtx.gz",
    "2.1.barcodes.tsv.gz": f"{BASE}/samples/GSM2560nnn/GSM2560248/suppl/GSM2560248_barcodes.tsv.gz",
    "2.2.mtx.gz":        f"{BASE}/samples/GSM2560nnn/GSM2560249/suppl/GSM2560249_2.2.mtx.gz",
    "2.2.barcodes.tsv.gz": f"{BASE}/samples/GSM2560nnn/GSM2560249/suppl/GSM2560249_barcodes.tsv.gz",
    "genes.tsv.gz":      f"{BASE}/series/GSE96nnn/GSE96583/suppl/GSE96583_batch2.genes.tsv.gz",
    "metadata.tsv.gz":   f"{BASE}/series/GSE96nnn/GSE96583/suppl/GSE96583_batch2.total.tsne.df.tsv.gz",
}

def descargar(url, destino):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=300) as r, open(destino, "wb") as f:
        f.write(r.read())

for nombre, url in FICHEROS.items():
    destino = DATA_DIR / nombre
    if not destino.exists():
        print(f"Descargando {nombre} ...", flush=True)
        try:
            descargar(url, destino)
        except Exception as e:
            print(f"Descarga fallida: {e}")
            print("-> Sube el fichero manualmente a la carpeta", DATA_DIR)
            raise

print("\nDescargado:")
for p in sorted(DATA_DIR.iterdir()):
    print(f"  {p.name:25s} {p.stat().st_size / 1e6:6.1f} MB")

Cargamos las dos matrices en formato Matrix Market (genes × células, dispersa) y construimos un `AnnData` por corrida. Los nombres de gen los hacemos únicos desde el principio (hay símbolos duplicados).

In [ ]:
genes = pd.read_csv(DATA_DIR / "genes.tsv.gz", sep="\t", header=None,
                    names=["ensembl", "symbol"], index_col=1)
genes.index.name = None
print("Genes:", len(genes))
genes.head()

In [ ]:
def cargar_corrida(mtx_path, barcodes_path, genes):
    """Lee una matriz 10x en formato Matrix Market -> AnnData (células x genes)."""
    X = scipy.io.mmread(mtx_path)      # genes x células
    X = sp.csr_matrix(X).T             # trasponemos: células x genes
    barcodes = pd.read_csv(barcodes_path, header=None)[0].values
    a = ad.AnnData(X, obs=pd.DataFrame(index=barcodes), var=genes.copy())
    a.var_names_make_unique()
    return a

a21 = cargar_corrida(DATA_DIR / "2.1.mtx.gz", DATA_DIR / "2.1.barcodes.tsv.gz", genes)
a22 = cargar_corrida(DATA_DIR / "2.2.mtx.gz", DATA_DIR / "2.2.barcodes.tsv.gz", genes)
print("Corrida 2.1 (ctrl):", a21.shape)
print("Corrida 2.2 (stim):", a22.shape)

## 2. Anotación de donantes y condiciones (demultiplexado)

Las dos corridas mezclan células de los 8 donantes. Kang *et al.* asignaron cada célula a su donante comparando sus variantes genéticas con los genotipos de los donantes (**demuxlet**), y también marcaron los **dobletes** (gotas con células de dos donantes distintos) y las gotas ambiguas.

Cruzamos nuestros barcodes con la tabla de metadatos de los autores. Ojo al detalle: algunos barcodes se repiten entre corridas, así que cruzamos cada corrida solo con el subconjunto de metadatos de su condición (`2.1` = `ctrl`, `2.2` = `stim`).

In [ ]:
meta = pd.read_csv(DATA_DIR / "metadata.tsv.gz", sep="\t", index_col=0)
meta.columns = ["tsne1", "tsne2", "donor", "stim", "cluster", "cell_type", "multiplets"]
print("Metadatos:", meta.shape)
print(meta["stim"].value_counts().to_dict())
meta.head()

In [ ]:
# Cruzamos cada corrida con su subconjunto de metadatos
meta_ctrl = meta[meta["stim"] == "ctrl"]
meta_stim = meta[meta["stim"] == "stim"]

a21 = a21[a21.obs_names.intersection(meta_ctrl.index)].copy()
a22 = a22[a22.obs_names.intersection(meta_stim.index)].copy()
a21.obs = a21.obs.join(meta_ctrl[["donor", "stim", "cell_type", "multiplets"]])
a22.obs = a22.obs.join(meta_stim[["donor", "stim", "cell_type", "multiplets"]])

adata = ad.concat([a21, a22], join="inner")
adata.obs_names_make_unique()

# Nos quedamos solo con singlets (células asignadas a un único donante)
adata = adata[adata.obs["multiplets"] == "singlet"].copy()

for c in ["donor", "stim", "cell_type"]:
    adata.obs[c] = adata.obs[c].astype(str).astype("category")

print(f"{adata.shape[0]} células x {adata.shape[1]} genes")
print("\nCélulas por donante y condición:")
pd.crosstab(adata.obs["donor"], adata.obs["stim"])

**Diseño equilibrado:** cada donante aporta células a las dos condiciones (diseño pareado) — ideal para el análisis diferencial.

## 3. Control de calidad

Repetimos el QC de la práctica 9. Al ser PBMC cultivados 6 h, toleramos un % mitocondrial algo mayor (<10 %) que en la práctica anterior.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

n0 = adata.n_obs
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[
    (adata.obs["n_genes_by_counts"] < 3000) & (adata.obs["pct_counts_mt"] < 10)
].copy()
print(f"QC: {n0} -> {adata.n_obs} células, {adata.n_vars} genes")

sc.pl.violin(adata, ["n_genes_by_counts", "pct_counts_mt"], groupby="stim", multi_panel=True)

## 4. Pseudobulk: la réplica correcta es el donante

Con ~6.000 monocitos por condición, sería tentador comparar célula a célula con un test de Wilcoxon. **Error estadístico grave**: las células de un mismo donante están correlacionadas entre sí (comparten genética, ambiente, procesado); tratarlas como réplicas independientes infla artificialmente la significación (*pseudoréplicas*; [Squair *et al.*, Nature Communications 2021](https://doi.org/10.1038/s41467-021-25960-2)).

La solución estándar es el análisis **pseudobulk**:

1. **Sumar** los conteos de todas las células de cada grupo `donante × condición × tipo celular` → una "muestra bulk" por combinación.
2. Analizar esas muestras con los métodos de bulk (DESeq2), donde la réplica es el **donante**.

Así recuperamos todo lo aprendido en las prácticas 3–4. Construimos la matriz pseudobulk ahora, con los conteos crudos y todos los genes, y la guardaremos para la sección 6.

In [ ]:
# Grupo pseudobulk = donante | condición | tipo celular
adata.obs["pb_group"] = (
    adata.obs["donor"].astype(str) + "|"
    + adata.obs["stim"].astype(str) + "|"
    + adata.obs["cell_type"].astype(str)
)

# Sumar conteos por grupo (matriz dispersa -> eficiente)
X_counts = sp.csr_matrix(adata.X)
categorias = pd.Categorical(adata.obs["pb_group"])
grupos = list(categorias.categories)
codes = categorias.codes

pb = np.zeros((len(grupos), adata.n_vars), dtype=np.float64)
for i in range(len(grupos)):
    pb[i, :] = np.asarray(X_counts[codes == i].sum(axis=0)).ravel()

pb_adata = ad.AnnData(pb, obs=pd.DataFrame(index=grupos), var=adata.var.copy())
pb_adata.obs[["donor", "stim", "cell_type"]] = (
    pb_adata.obs_names.to_series().str.split("|", expand=True).values
)
pb_adata.obs["n_cells"] = pd.Series(codes).value_counts().sort_index().values

print("Matriz pseudobulk:", pb_adata.shape[0], "muestras x", pb_adata.shape[1], "genes")
print("\nMuestras por tipo celular y condición:")
print(pb_adata.obs.groupby(["cell_type", "stim"]).size().unstack(fill_value=0))
print("\nRango de células por muestra:", int(pb_adata.obs["n_cells"].min()),
      "-", int(pb_adata.obs["n_cells"].max()))

Tenemos **128 muestras pseudobulk** (8 donantes × 2 condiciones × 8 tipos celulares). Cada una suma los conteos de cientos de células.

> **Pregunta 1.** ¿Por qué sumamos conteos crudos y no promediamos la matriz normalizada? Pista: ¿qué modelo estadístico usa DESeq2 y sobre qué tipo de datos?

## 5. Integración de muestras con Harmony

Antes del análisis diferencial, exploremos el dataset completo. Seguimos el flujo de la práctica 9 (normalizar → HVG → escalar → PCA → vecinos → UMAP) y coloreamos por condición, donante y tipo celular.

In [ ]:
# Normalización y selección de HVG (por lote, para no confundir batch con variabilidad)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata  # guardamos la matriz normalizada completa (para la sección 7)

sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="stim")
print("HVG:", int(adata.var["highly_variable"].sum()))

adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")

In [ ]:
# UMAP SIN integrar
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
sc.pl.umap(adata, color=["stim", "donor", "cell_type"], ncols=3, wspace=0.4)

**¿Qué vemos?**

- Los **donantes** están bien mezclados desde el principio: todas las células se procesaron juntas en solo dos corridas (¡el multiplexado genético es, de por sí, un diseño anti-batch!).
- Lo que separa a las células es la **condición**: dentro de cada tipo celular aparecen dos sub-nubes (control y estimulada). Eso es **biología**, no batch: es la respuesta al interferón, y no queremos eliminarla.

Aun así, aplicamos **Harmony** sobre el PCA usando el donante como variable de batch para alinear los perfiles residuales de cada donante. Harmony ajusta las coordenadas PCA de forma iterativa para que las células de distintos batches ocupen la misma región del espacio, **sin tocar** los datos de expresión.

In [ ]:
import scanpy.external as sce

sce.pp.harmony_integrate(adata, key="donor", adjusted_basis="X_pca_harmony")

# Rehacemos vecinos y UMAP sobre el PCA corregido
sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
sc.pl.umap(adata, color=["stim", "donor", "cell_type"], ncols=3, wspace=0.4)

In [ ]:
# Clustering de Leiden sobre el espacio integrado
sc.tl.leiden(adata, resolution=0.5, key_added="leiden")

# ¿Coinciden nuestros clústeres con la anotación de los autores?
tabla = pd.crosstab(adata.obs["leiden"], adata.obs["cell_type"])
print(tabla)
sc.pl.umap(adata, color=["leiden", "cell_type"], wspace=0.4)

> **Pregunta 2.** Compara los UMAP antes y después de Harmony: ¿qué variable se mezcla mejor? ¿Por qué **no** debemos integrar usando la condición (`stim`) como batch? ¿Qué pasaría con la respuesta al interferón?

> **Pregunta 3.** Muchos clústeres de Leiden aparecen "duplicados" por parejas (mismo tipo celular, dos condiciones). ¿Por qué el clustering separa por condición dentro de un mismo tipo? ¿Es un problema de la integración o una señal real?


## 6. Ejercicio de anotación: ¿reproducimos los tipos celulares de los autores?

Hasta ahora hemos usado la anotación de tipo celular de Kang *et al.*. Pero en un proyecto real esa tabla no existiría: la anotación sería **nuestro trabajo**, como en la práctica 9. Hagámoslo aquí como ejercicio.

> **Ejercicio 1.** Usa el dotplot con los marcadores canónicos de la práctica 9 para asignar cada clúster de Leiden a un tipo celular. Completa el diccionario `mi_anotacion` y comprueba qué fracción de células coincide con la anotación oficial. Ojo: hay 14 clústeres para 8 tipos — recuerda la **pregunta 3** (la mayoría de tipos aparecen duplicados como pareja control/estimulada).

In [ ]:
marcadores_canonicos = {
    "T CD4": ["IL7R", "CCR7"],
    "T CD8": ["CD8A", "CD8B"],
    "NK": ["GNLY", "NKG7"],
    "B": ["MS4A1", "CD79A"],
    "Mono CD14+": ["CD14", "LYZ", "S100A9"],
    "Mono FCGR3A+": ["FCGR3A", "MS4A7"],
    "Dendríticas": ["FCER1A", "CST3"],
    "Megacariocitos": ["PPBP", "PF4"],
}
sc.pl.dotplot(adata, marcadores_canonicos, groupby="leiden", use_raw=True)

In [ ]:
# Solución orientativa — complétala tú primero mirando el dotplot
mi_anotacion = {'0': 'CD4 T cells', '1': 'CD4 T cells', '2': 'CD14+ Monocytes', '3': 'CD14+ Monocytes', '4': 'B cells', '5': 'NK cells', '6': 'CD8 T cells', '7': 'CD4 T cells', '8': 'FCGR3A+ Monocytes', '9': 'FCGR3A+ Monocytes', '10': 'Megakaryocytes', '11': 'Dendritic cells', '12': 'Dendritic cells', '13': 'CD4 T cells'}

adata.obs["mi_anotacion"] = adata.obs["leiden"].map(mi_anotacion)

acuerdo = (adata.obs["mi_anotacion"] == adata.obs["cell_type"]).mean()
print(f"Acuerdo con la anotación de los autores: {acuerdo:.1%}")

# ¿Dónde discrepa nuestra anotación de la oficial?
pd.crosstab(adata.obs["mi_anotacion"], adata.obs["cell_type"])

Un acuerdo del ~85–95 % es lo esperable: los marcadores canónicos resuelven bien los tipos principales, y las discrepancias se concentran en subtipos cercanos (p. ej. separar T CD4 de T CD8, o los dos tipos de monocitos) y en clústeres pequeños.

## 7. Composición celular: ¿cambia la proporción de tipos con el estímulo?

Antes de preguntarnos qué genes cambian **dentro** de cada tipo, preguntémonos si cambia la **composición**: ¿el IFN-β expande o contrae algún tipo celular? El diseño pareado (cada donante aporta células a las dos condiciones) permite comparar proporciones donante a donante.

**Aviso metodológico:** las proporciones son datos **composicionales** — suman 1, de modo que si un tipo sube, los demás bajan mecánicamente aunque no les pase nada. Su análisis riguroso requiere métodos específicos (p. ej. [scCODA](https://doi.org/10.1038/s41467-021-27150-6)). Aquí haremos una exploración visual: con 8 donantes pareados, los cambios fuertes se ven a simple vista.

In [ ]:
props = (adata.obs.groupby(["donor", "stim", "cell_type"], observed=True)
         .size().rename("n").reset_index())
props["prop"] = props["n"] / props.groupby(["donor", "stim"], observed=True)["n"].transform("sum")

# Cambio medio (stim - ctrl) en puntos porcentuales por tipo celular
resumen_prop = props.groupby(["cell_type", "stim"], observed=True)["prop"].mean().unstack()
resumen_prop["delta_pp"] = (resumen_prop["stim"] - resumen_prop["ctrl"]) * 100
resumen_prop.sort_values("delta_pp").round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
tipos = resumen_prop.sort_values("ctrl", ascending=False).index.tolist()

for i, ct in enumerate(tipos):
    sub = props[props["cell_type"] == ct].pivot(index="donor", columns="stim", values="prop")
    for donor, row in sub.iterrows():
        ax.plot([i - 0.1, i + 0.1], [row["ctrl"], row["stim"]], "-o",
                color="lightgrey", markerfacecolor="grey", markersize=3.5, lw=1, zorder=1)
    ax.plot(i - 0.1, sub["ctrl"].mean(), "o", color="steelblue", markersize=9, zorder=2)
    ax.plot(i + 0.1, sub["stim"].mean(), "o", color="firebrick", markersize=9, zorder=2)

ax.plot([], [], "o", color="steelblue", label="ctrl (media)")
ax.plot([], [], "o", color="firebrick", label="stim (media)")
ax.plot([], [], "-o", color="grey", markersize=3.5, lw=1, label="donantes")
ax.set_xticks(range(len(tipos)))
ax.set_xticklabels(tipos, rotation=45, ha="right")
ax.set_ylabel("Proporción de células")
ax.set_title("Composición celular por donante en cada condición")
ax.legend()
plt.tight_layout()
plt.show()

Las proporciones apenas se mueven (el mayor cambio es de ~2 puntos porcentuales en monocitos CD14+). **La respuesta al IFN-β es transcripcional, no composicional**: los tipos celulares no se expanden ni desaparecen; lo que hacen es reprogramar masivamente su expresión génica. Eso es exactamente lo que cuantificamos en la siguiente sección.

> **Pregunta 4.** Si la composición apenas cambia pero en el UMAP cada tipo celular se separaba en dos nubes por condición, ¿dónde reside entonces la señal del interferón?

## 8. Expresión diferencial pseudobulk con PyDESeq2

Volvamos a la pregunta biológica: **¿qué genes responden al IFN-β en monocitos CD14+?** Recuperamos la matriz pseudobulk de la sección 4, nos quedamos con las 16 muestras de monocitos (8 donantes × 2 condiciones) y aplicamos DESeq2 exactamente como en las prácticas 3–4.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# Muestras de monocitos CD14+
mono = pb_adata[pb_adata.obs["cell_type"] == "CD14+ Monocytes"].copy()
mono = mono[:, mono.X.sum(axis=0) > 0].copy()
mono.X = mono.X.astype(int)

print("Muestras:", mono.n_obs, "| genes:", mono.n_vars)
mono.obs[["donor", "stim", "n_cells"]]

In [ ]:
dds = DeseqDataSet(adata=mono, design="~stim", refit_cooks=True)
dds.deseq2()

ds = DeseqStats(dds, contrast=["stim", "stim", "ctrl"])
ds.summary()

res = ds.results_df
res.head(10)[["baseMean", "log2FoldChange", "pvalue", "padj"]]

In [ ]:
n_sig = ((res["padj"] < 0.05) & (res["log2FoldChange"].abs() > 1)).sum()
print(f"Genes significativos (FDR < 0.05 y |log2FC| > 1): {n_sig}")
print("\nTop 15 por significación:")
res.sort_values("padj").head(15)[["log2FoldChange", "padj"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
sig = (res["padj"] < 0.05) & (res["log2FoldChange"].abs() > 1)
ax.scatter(res["log2FoldChange"], -np.log10(res["padj"]), s=3, c="lightgrey", alpha=0.5)
ax.scatter(res.loc[sig, "log2FoldChange"], -np.log10(res.loc[sig, "padj"]),
           s=4, c="firebrick", alpha=0.6)

# Etiquetamos algunos genes estimulados por interferón (ISG) clásicos
isgs = ["IFITM1", "IFITM2", "IFITM3", "ISG15", "ISG20", "MX1", "MX2",
        "OAS1", "OAS2", "OAS3", "IFIT1", "IFIT2", "IFIT3", "DDX58", "STAT1"]
for g in isgs:
    if g in res.index and sig.get(g, False):
        ax.annotate(g, (res.loc[g, "log2FoldChange"], -np.log10(res.loc[g, "padj"])),
                    fontsize=7, alpha=0.9)

ax.axhline(-np.log10(0.05), ls="--", c="grey", lw=0.8)
ax.axvline(1, ls="--", c="grey", lw=0.8)
ax.axvline(-1, ls="--", c="grey", lw=0.8)
ax.set_xlabel("log2 Fold Change (stim vs ctrl)")
ax.set_ylabel("-log10(FDR)")
ax.set_title("Respuesta al IFN-β en monocitos CD14+ (pseudobulk, n = 8 donantes)")
plt.show()

El volcán está dominado por los **genes estimulados por interferón (ISG)**: `IFIT`, `IFITM`, `ISG`, `MX`, `OAS`, `DDX58` (RIG-I)... La respuesta antiviral de tipo I en toda su gloria, y con soporte estadístico correcto (8 réplicas biológicas por grupo).

> **Ejercicio 2.** Repite el análisis pseudobulk para las **células NK** (`NK cells`). ¿Cómo cambia la respuesta? ¿Es más fuerte o más débil que en monocitos?

## 9. ¿Y si hubiéramos hecho el test célula a célula? (pseudoréplicas)

Comparemos con el análisis **ingenuo**: Wilcoxon entre las ~6.000 células de monocitos de cada condición, como hicimos por clúster en la práctica 9.

In [ ]:
# Análisis "ingenuo": cada célula como si fuera una réplica independiente
mono_cells = adata[adata.obs["cell_type"] == "CD14+ Monocytes"].copy()
sc.tl.rank_genes_groups(mono_cells, "stim", method="wilcoxon", use_raw=True)
naive = sc.get.rank_genes_groups_df(mono_cells, group="stim")

n_naive = (naive["pvals_adj"] < 0.05).sum()
n_pb = (res["padj"] < 0.05).sum()
print(f"Significativos (solo FDR < 0.05):")
print(f"  Test célula a célula ({mono_cells.n_obs} 'réplicas'): {n_naive}")
print(f"  Pseudobulk DESeq2 (16 réplicas reales):            {n_pb}")
print(f"\nFDR ajustado más pequeño posible con cada aproximación:")
print(f"  Wilcoxon por célula: {naive['pvals_adj'].min():.2e}")
print(f"  DESeq2 pseudobulk:   {res['padj'].min():.2e}")

In [ ]:
# ¿Coinciden las magnitudes del efecto?
comun = (naive.set_index("names")[["logfoldchanges"]]
         .join(res[["log2FoldChange", "baseMean"]], how="inner").dropna())
expresados = comun[comun["baseMean"] > 50]
r = expresados["logfoldchanges"].corr(expresados["log2FoldChange"])

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(expresados["log2FoldChange"], expresados["logfoldchanges"], s=4, alpha=0.4)
ax.set_xlabel("log2FC pseudobulk (DESeq2)")
ax.set_ylabel("log2FC célula a célula (Wilcoxon)")
ax.set_title(f"Misma señal biológica (r = {r:.3f}, genes expresados)")
plt.show()

**Lección:** la señal biológica es la misma (los log2FC coinciden, r ≈ 0,99 en genes expresados). Lo que cambia es la **inferencia**:

- El test célula a célula asume ~6.000 observaciones **independientes** → p-valores de confianza excesiva cuando las células de un donante están correlacionadas.
- El pseudobulk usa las **16 réplicas reales** y modela la varianza entre donantes → p-valores calibrados.

En este dataset los donantes están muy bien mezclados, así que las conclusiones coinciden; en datos de pacientes (con heterogeneidad real entre individuos) el análisis célula a célula produce falsos positivos masivos. Por eso el consenso actual ([Squair *et al.* 2021](https://doi.org/10.1038/s41467-021-25960-2)) es: **DE entre condiciones → pseudobulk; DE entre clústeres dentro de una muestra → test por célula** (como en la práctica 9).

## 10. (Opcional) Análisis funcional con decoupler

Cerramos el círculo con la práctica 7: tomamos el estadístico de DESeq2 de cada gen y puntuamos las vías **Hallmark** con ULM. ¿Qué programas se activan en los monocitos estimulados?

In [ ]:
import decoupler as dc

hallmark = dc.op.hallmark(organism="human")

# Matriz de entrada: una fila por contraste, una columna por gen (estadístico de Wald)
mat = res[["stat"]].dropna().T.rename(index={"stat": "stim.vs.ctrl"})
mat = mat.loc[:, mat.columns.notna() & (mat.columns != "")]
mat = mat.groupby(mat.columns, axis=1).mean()  # agregar símbolos duplicados

acts, pvals = dc.mt.ulm(data=mat, net=hallmark, tmin=5)

resultado = acts.T.join(pvals.T.add_suffix("_pval")).sort_values("stim.vs.ctrl", ascending=False)
resultado.head(10)

In [ ]:
import matplotlib.pyplot as plt

top = resultado.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index, top["stim.vs.ctrl"], color="steelblue")
ax.set_xlabel("Score de actividad (ULM)")
ax.set_title("Vías Hallmark activadas por IFN-β en monocitos CD14+")
plt.tight_layout()
plt.show()

Los dos programas más activados son **INTERFERON_GAMMA_RESPONSE** e **INTERFERON_ALPHA_RESPONSE**: exactamente la biología esperada tras estimular con un interferón de tipo I. El pipeline bulk → pseudobulk → análisis funcional queda así conectado de punta a punta.

## Resumen

| Paso | Herramienta | Idea clave |
|---|---|---|
| Carga multimuestra | `scipy.io.mmread` + `AnnData` | una matriz por corrida, metadatos de los autores |
| QC | `calculate_qc_metrics` | mismo criterio que en práctica 9 |
| Integración | Harmony (`donor` como batch) | alinear batches sin borrar la condición |
| Anotación | marcadores canónicos (práctica 9) | los clústeres reproducen los tipos oficiales |
| Composición | proporciones por donante | la respuesta es transcripcional, no composicional |
| Pseudobulk | suma de conteos por donante × condición × tipo | la réplica real es el donante |
| DE | PyDESeq2 sobre pseudobulk | las prácticas 3–4, otra vez |
| Funcional | decoupler + Hallmark | las prácticas 5 y 7, otra vez |

**Cierre del módulo:** hemos recorrido el arco completo — del bulk a la célula única y de vuelta al bulk (pseudobulk) para la inferencia rigurosa. Cada técnica responde a una pregunta distinta: composición (deconvolución, práctica 8), identidad celular (clustering, práctica 9), respuesta a estímulos (pseudobulk, práctica 10).

> **Pregunta final.** El diseño de Kang *et al.* multiplexó los donantes en las mismas corridas. ¿Qué problema experimental evita este diseño que sí tendría un estudio con una corrida 10x por paciente? Relaciónalo con el concepto de **confusión batch–condición** que vimos en el módulo.